# 03 — Feature engineering

Dùng đặc trưng số đã trích xuất, category hành chính/loại bất động sản và TF-IDF cho mô tả. `ColumnTransformer` giữ toàn bộ phép impute, one-hot và vector hóa bên trong pipeline để không fit trước trên tập kiểm tra.

Notebook minh họa cùng một mẫu cố định 5.000 dòng mà bước huấn luyện sử dụng.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src" / "pipeline.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Không tìm thấy thư mục dự án có src/pipeline.py")
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from src.pipeline import (
    FEATURE_COLUMNS,
    PUBLIC_SAMPLE_COLUMNS,
    TRAIN_SAMPLE_SIZE,
    load_processed_data,
    select_training_sample,
)

full_data = load_processed_data()
data = select_training_sample(full_data)
print(f"Mẫu huấn luyện: {len(data):,}/{len(full_data):,} dòng; seed=42")
print("Đặc trưng dùng trong model:", FEATURE_COLUMNS)
print("Cột được phép xuất trong bản demo:", PUBLIC_SAMPLE_COLUMNS)
display(data[[
    "price_bil", "area_m2", "bedrooms", "floors", "frontage",
    "district", "property_type",
]].head(10))

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=1, max_features=8000,
    sublinear_tf=True, token_pattern=r"(?u)\b\w+\b",
)
tfidf = vectorizer.fit_transform(
    data["text"].fillna("khong mo ta").astype(str)
)
print(f"Ma trận TF-IDF: {tfidf.shape[0]:,} tin × {tfidf.shape[1]:,} từ/cụm từ")
print(f"Dữ liệu demo xuất ra: {PROJECT_ROOT / 'data' / 'demo'}")


Mẫu huấn luyện: 5,000/51,132 dòng; seed=42
Đặc trưng dùng trong model: ['area_m2', 'bedrooms', 'floors', 'frontage', 'district', 'ward', 'property_type', 'text']
Cột được phép xuất trong bản demo: ['price_bil', 'area_m2', 'bedrooms', 'floors', 'frontage', 'district', 'property_type']


,price_bil,area_m2,bedrooms,floors,frontage,district,property_type
0,7.0,115.0,NaN,2.0,NaN,Q12,"kho, nha xuong"
1,6.8,96.0,NaN,5.0,5.00,Go_Vap,dat
2,2.1,84.0,NaN,NaN,5.00,Binh_Chanh,dat
3,6.5,66.0,NaN,NaN,NaN,TP_Thu_Duc,dat
4,69.0,4000.0,NaN,2.0,NaN,Binh_Chanh,"kho, nha xuong"
5,1.4,130.0,NaN,NaN,5.00,Cu_Chi,dat
6,16.5,192.0,NaN,6.0,8.00,Q9,dat
7,2.9,21.0,2.0,1.0,3.03,Tan_Phu,nha rieng
8,7.3,691.1,NaN,NaN,NaN,__missing__,nha rieng
9,4.0,71.0,2.0,NaN,NaN,Nha_Be,can ho chung cu


Ma trận TF-IDF: 5,000 tin × 8,000 từ/cụm từ
Dữ liệu demo xuất ra: D:\Dowload\Setup project\DuAnDuBaoGiaNhaTPHCM\data\demo


## Tách tập train/test

Giữ 20% mẫu làm tập kiểm tra; mọi phép biến đổi học tham số trong pipeline trên phần train.

In [2]:
from sklearn.model_selection import train_test_split

sample_train, sample_test = train_test_split(
    data, test_size=0.2, random_state=42
)
print(f"Train: {len(sample_train):,} dòng")
print(f"Test:  {len(sample_test):,} dòng")
print("Không dùng giá, ID tin, tọa độ hoặc nội dung liên hệ làm đầu vào trực tiếp.")


Train: 4,000 dòng
Test:  1,000 dòng
Không dùng giá, ID tin, tọa độ hoặc nội dung liên hệ làm đầu vào trực tiếp.
